# Train the PI-NGB-TSTR models with the official libraries and export them for the app

This notebook trains the three physics-informed NGBoost models (ITSD, TSR, RD) **exactly as in your PI-NGB-TSTR notebooks**
(same data files, physics descriptors, preprocessing, hyperparameters, seeds, TSR input selection and sample weights), using
the library versions of the published models (**scikit-learn 1.6.1, ngboost 0.5.11**). It then exports each model to the
app's portable format and checks that the exported model reproduces `model.predict` exactly.

**You need:** `2. TSTR_ITSD_RAPSuperpave.xlsx`, `4. TSTR_TSR_RAPSuperpave_resplit.xlsx`, `6. TSTR_RD_RAPSuperpave_resplit.xlsx`.

**You get:** `models_official.zip` (copy `ITSD.npz`, `TSR.npz`, `RD.npz` into the app's `models/` folder, replacing the interim
files) and `PI-NGB_TSTR_pickles.zip` (the trained models and preprocessors as `.pkl`, with corrected file names).

*Runtime → Run all.* Training takes about 2–5 minutes.

## 1. Install the library versions of the published models

In [ ]:
!pip install -q "scikit-learn==1.6.1" "ngboost==0.5.11"
import sklearn, ngboost, numpy, pandas
print('scikit-learn', sklearn.__version__, '| ngboost', ngboost.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)
if sklearn.__version__ != '1.6.1':
    print('\n>>> scikit-learn was already loaded in this session. Use Runtime -> Restart session, then run all cells again.')

## 2. App library (physics descriptors, prediction engine, exporter)

In [ ]:
import os; os.makedirs("rapcore", exist_ok=True)

In [ ]:
%%writefile rapcore/__init__.py
"""Core library of the RAP Superpave Performance Predictor (PI-NGB-TSTR)."""
from .features import INPUTS, TARGETS, prepare_inputs, check_inputs, physics_table
from .engine import Predictor

In [ ]:
%%writefile rapcore/features.py
"""Mixture inputs, physics-informed descriptors and input checks.

The descriptor formulas are copied from the PI-NGB-TSTR training notebooks so that the app computes
exactly the features the models were trained on."""
import numpy as np
import pandas as pd

TARGETS = ['ITSD', 'TSR', 'RD']
SIEVES = ['19 mm', '12.5 mm', '9.5 mm', '4.75 mm', '2.36 mm', '0.6 mm', '0.15 mm', '0.075 mm']
SIEVE_SIZES = np.array([19.0, 12.5, 9.5, 4.75, 2.36, 0.6, 0.15, 0.075])
PHYSICS = ['Gradation_n', 'Gradation_RMSE', 'Gradation_R2', 'AV_VMA_Ratio', 'VFA_Calc', 'VFA_Residual',
           'Volumetric_Consistency', 'Volumetric_Consistency_Error', 'AV_VMA_Violation']

# Inputs as in Table 2 of the paper: column, symbol, description, unit, category, kind ('num' or 'cat')
CATEGORIES = ['Aggregate gradation', 'Aggregate characteristics', 'Recyclable materials and binder properties',
              'Mixture volumetrics', 'Experimental protocols']
INPUTS = [
    *[(s, s, f'Passing the {s} sieve', '%', 'Aggregate gradation', 'num') for s in SIEVES],
    ('NMAS', 'NMAS', 'Nominal maximum aggregate size', 'mm', 'Aggregate characteristics', 'num'),
    ('LAA', 'LAA', 'Los Angeles abrasion loss', '%', 'Aggregate characteristics', 'num'),
    ('AT', 'AT', 'Aggregate type', '–', 'Aggregate characteristics', 'cat'),
    ('RAP', 'RAP', 'Reclaimed asphalt pavement content', '%', 'Recyclable materials and binder properties', 'num'),
    ('RAPPG', 'RAPPG', 'RAP binder performance grade', '–', 'Recyclable materials and binder properties', 'cat'),
    ('RT', 'RJ', 'Rejuvenator', '–', 'Recyclable materials and binder properties', 'cat'),
    ('VAPG', 'VAPG', 'Virgin asphalt performance grade', '–', 'Recyclable materials and binder properties', 'cat'),
    ('OBC', 'OBC', 'Optimum binder content', '%', 'Mixture volumetrics', 'num'),
    ('Gmm', 'Gmm', 'Theoretical maximum specific gravity', '–', 'Mixture volumetrics', 'num'),
    ('AV', 'AV', 'Air voids', '%', 'Mixture volumetrics', 'num'),
    ('VMA', 'VMA', 'Voids in mineral aggregate', '%', 'Mixture volumetrics', 'num'),
    ('VFA', 'VFA', 'Voids filled with asphalt', '%', 'Mixture volumetrics', 'num'),
    ('Ndes', 'Ndes', 'Number of design gyrations', '–', 'Experimental protocols', 'num'),
    ('RCT', 'RCT', 'Rutting cycle type', '–', 'Experimental protocols', 'cat'),
]
OUTPUTS = [('ITSD', 'Dry indirect tensile strength', 'kPa'), ('TSR', 'Tensile strength ratio', '–'), ('RD', 'Rut depth', 'mm')]
INPUT_NAMES = [i[0] for i in INPUTS]
CATEGORICAL = [i[0] for i in INPUTS if i[5] == 'cat']
NUMERIC = [i[0] for i in INPUTS if i[5] == 'num']
SYMBOL = {i[0]: i[1] for i in INPUTS}
DESC = {i[0]: i[2] for i in INPUTS}
UNITS = {i[0]: i[3] for i in INPUTS}
CATEGORY = {i[0]: i[4] for i in INPUTS}
LABELS = {c: (DESC[c] if c in SIEVES else f'{SYMBOL[c]} – {DESC[c]}') for c in INPUT_NAMES}
ALIASES = {'RJ': 'RT'}          # paper symbol -> database column


def form_label(c):
    """'Symbol – description (unit)' as in Table 2."""
    return f'{SYMBOL[c]} – {DESC[c]} ({UNITS[c]})'


def gradation_features(data):
    """Power-law (maximum-density) fit of the gradation: exponent n, fit RMSE and fit R² (notebook code)."""
    data = data.copy(); n_values, rmse_values, r2_values = [], [], []
    for _, row in data.iterrows():
        g = pd.to_numeric(row[SIEVES], errors='coerce').values.astype(float)
        valid = np.isfinite(g) & (g > 0) & (g <= 100)
        if np.sum(valid) >= 2:
            x = np.log(SIEVE_SIZES[valid] / 19.0); y = np.log(g[valid] / 100.0)
            den = np.sum(x * x)
            if den > 1e-12:
                n = np.clip(np.sum(x * y) / den, 0.05, 2.0)
                pred = 100.0 * (SIEVE_SIZES[valid] / 19.0) ** n; obs = g[valid]
                rmse = np.sqrt(np.mean((obs - pred) ** 2))
                ss_res = np.sum((obs - pred) ** 2); ss_tot = np.sum((obs - np.mean(obs)) ** 2)
                r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else 0.0
            else:
                n, rmse, r2 = 0.45, np.nan, np.nan
        else:
            n, rmse, r2 = 0.45, np.nan, np.nan
        n_values.append(n); rmse_values.append(rmse); r2_values.append(r2)
    data['Gradation_n'] = n_values; data['Gradation_RMSE'] = rmse_values; data['Gradation_R2'] = r2_values
    return data


def volumetric_features(data):
    """Superpave volumetric relationships AV–VMA–VFA (notebook code)."""
    data = data.copy()
    av, vma, vfa = (pd.to_numeric(data[c], errors='coerce') for c in ('AV', 'VMA', 'VFA'))
    safe_vma = np.maximum(vma.values, 1e-8)
    data['AV_VMA_Ratio'] = av.values / safe_vma
    vfa_calc = 100.0 * (vma.values - av.values) / safe_vma
    data['VFA_Calc'] = vfa_calc
    data['VFA_Residual'] = vfa.values - vfa_calc
    data['Volumetric_Consistency'] = vfa.values / 100.0 + av.values / safe_vma
    data['Volumetric_Consistency_Error'] = np.abs(data['Volumetric_Consistency'] - 1.0)
    data['AV_VMA_Violation'] = (av.values > vma.values).astype(int)
    return data


def prepare_inputs(df):
    """Raw mixture table -> table with the nine physics descriptors added (same order as in training)."""
    df = df.copy()
    for c in NUMERIC:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in CATEGORICAL:
        if c in df:
            df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)
    return volumetric_features(gradation_features(df))


def physics_table(df_prepared, i=0):
    """Descriptor values of one mixture, with plain-language names, for display."""
    names = {'Gradation_n': 'Gradation exponent n (power-law fit)', 'Gradation_RMSE': 'Gradation fit RMSE (% passing)',
             'Gradation_R2': 'Gradation fit R²', 'AV_VMA_Ratio': 'AV/VMA ratio', 'VFA_Calc': 'Calculated VFA (%)',
             'VFA_Residual': 'VFA residual (entered − calculated, %)', 'Volumetric_Consistency': 'Volumetric consistency (ideal = 1)',
             'Volumetric_Consistency_Error': 'Volumetric consistency error', 'AV_VMA_Violation': 'AV > VMA flag'}
    r = df_prepared.iloc[i]
    return pd.DataFrame({'Descriptor': [names[p] for p in PHYSICS], 'Value': [r[p] for p in PHYSICS]})


def check_inputs(row, domain):
    """Engineering checks of one mixture. Returns a list of (level, message); level is 'error' or 'warning'.
    `domain` holds the ranges and categories of the 189 experimental mixtures used to build the models."""
    msgs = []
    g = np.array([row.get(s, np.nan) for s in SIEVES], float)
    missing = [LABELS[c] for c in INPUT_NAMES if (pd.isna(row.get(c, np.nan)) or row.get(c, '') == '')]
    if missing:
        msgs.append(('warning', 'Missing inputs were filled with the training median or most frequent value: ' + ', '.join(missing) + '.'))
    if np.any((g < 0) | (g > 100)):
        msgs.append(('error', 'Percent passing must lie between 0 and 100 for every sieve.'))
    gg = g[np.isfinite(g)]
    if len(gg) > 1 and np.any(np.diff(gg) > 1e-9):
        msgs.append(('error', 'Gradation must not increase from coarser to finer sieves (percent passing must decrease or stay equal).'))
    av, vma, vfa = (row.get(c, np.nan) for c in ('AV', 'VMA', 'VFA'))
    if np.isfinite(av) and np.isfinite(vma):
        if av >= vma:
            msgs.append(('error', 'Air voids must be smaller than VMA (AV < VMA).'))
        elif np.isfinite(vfa):
            vc = 100 * (vma - av) / vma
            if abs(vfa - vc) > 3:
                msgs.append(('warning', f'VFA ({vfa:.1f}%) is inconsistent with AV and VMA; the Superpave identity gives {vc:.1f}%.'))
    out = []
    for c in NUMERIC:
        v = row.get(c, np.nan)
        lo, hi = domain['ranges'][c]
        if np.isfinite(v) and (v < lo - 1e-9 or v > hi + 1e-9):
            u = '' if UNITS[c] == '–' else f' {UNITS[c]}'
            out.append(f'{LABELS[c]} = {v:g}{u} (database range {lo:g}–{hi:g}{u})')
    for c in CATEGORICAL:
        v = row.get(c, None)
        if isinstance(v, str) and v not in domain['categories'][c]:
            out.append(f'{LABELS[c]} = {v} (not in the database)')
    if out:
        msgs.append(('warning', 'Outside the experimental database, so the prediction is an extrapolation and its interval may be '
                                'too narrow: ' + '; '.join(out) + '.'))
    return msgs

In [ ]:
%%writefile rapcore/engine.py
"""Prediction engine for the exported PI-NGB-TSTR models.

Each model file (models/<TARGET>.npz) holds the NGBoost tree ensemble as plain arrays, the one-hot
encoding of the categorical inputs, the training fill values, a background sample for SHAP and the
split-conformal calibration scores. Only NumPy is needed to evaluate it, so the app does not depend on
the scikit-learn or ngboost versions used for training."""
import json, math, os
import numpy as np
import pandas as pd
from .features import prepare_inputs, check_inputs, TARGETS

TSR_LIMIT = 0.80


class NGBEnsemble:
    def __init__(self, path):
        z = np.load(path, allow_pickle=False)
        self.spec = json.loads(str(z['spec']))
        for k in ('left', 'right', 'feat', 'thr', 'val', 'roots', 'tree_param', 'tree_mult', 'bg'):
            setattr(self, k, z[k])
        s = self.spec
        self.target, self.dist = s['target'], s['dist']
        self.init = np.array(s['init_params'], float)
        self.W = np.zeros((len(self.roots), 2)); self.W[np.arange(len(self.roots)), self.tree_param] = self.tree_mult
        self.groups = self._groups()

    # ---------------------------------------------------------------- preprocessing (= notebook ColumnTransformer)
    def design_matrix(self, prepared):
        s = self.spec; X = prepared.copy()
        for c in s['x_cols']:
            if c not in X:
                X[c] = np.nan
            X[c] = X[c].where(X[c].notna(), s['fill'].get(c, np.nan))
        blocks = []
        for c in s['cat_cols']:
            cats, drop = s['categories'][c], s['drop_idx'][c]
            v = X[c].astype(str).values
            for k, cat in enumerate(cats):
                if drop is not None and k == drop:
                    continue
                blocks.append((v == cat).astype(float))
        for c in s['num_cols']:
            blocks.append(pd.to_numeric(X[c], errors='coerce').values.astype(float))
        return np.column_stack(blocks)

    def _groups(self):
        """Processed columns belonging to each original input (one SHAP player per input)."""
        s, g, j = self.spec, [], 0
        for c in s['cat_cols']:
            n = len(s['categories'][c]) - (s['drop_idx'][c] is not None)
            g.append((c, np.arange(j, j + n))); j += n
        for c in s['num_cols']:
            g.append((c, np.array([j]))); j += 1
        return g

    # ---------------------------------------------------------------- NGBoost: parameters and mean
    def params(self, M, chunk=1500):
        M = np.asarray(M, float).astype(np.float32).astype(np.float64)   # scikit-learn trees split on float32 inputs
        out = np.empty((len(M), 2))
        T = len(self.roots); depth = int(self.spec['max_depth'])
        for a in range(0, len(M), chunk):
            X = M[a:a + chunk]
            idx = np.broadcast_to(self.roots, (len(X), T)).copy()
            for _ in range(depth):
                xv = np.take_along_axis(X, self.feat[idx], axis=1)
                idx = np.where(xv <= self.thr[idx], self.left[idx], self.right[idx])
            out[a:a + chunk] = self.init - self.val[idx] @ self.W
        return out

    def mean(self, M):
        P = self.params(M); loc, scale = P[:, 0], np.exp(P[:, 1])
        return np.exp(loc + scale ** 2 / 2) if self.dist == 'lognormal' else loc

    # ---------------------------------------------------------------- split-conformal interval
    def interval(self, mu, level=0.90):
        cal = self.spec['calibration']; sc = np.sort(np.array(cal['scores'], float)); n = len(sc)
        k = min(n, int(math.ceil((n + 1) * level))); q = sc[k - 1]
        if cal['mode'] == 'relative':
            return np.maximum(mu * (1 - q), 0.0), mu * (1 + q)
        return mu - q, mu + q

    # ---------------------------------------------------------------- SHAP (interventional, antithetic permutations)
    def shap(self, m_row, n_perm=10, seed=0):
        rng = np.random.default_rng(seed); BG = self.bg; P = len(self.groups)
        perms = []
        for _ in range(max(1, n_perm // 2)):
            p = rng.permutation(P); perms += [p, p[::-1]]
        rows = []
        for perm in perms:
            cur = BG.copy(); rows.append(cur.copy())
            for j in perm:
                cur[:, self.groups[j][1]] = m_row[self.groups[j][1]]; rows.append(cur.copy())
        out = self.mean(np.vstack(rows)).reshape(len(perms), P + 1, len(BG)).mean(axis=2)
        phi = np.zeros(P)
        for k, perm in enumerate(perms):
            phi[perm] += np.diff(out[k])
        phi /= len(perms)
        base = out[:, 0].mean()
        return pd.Series(phi, index=[g[0] for g in self.groups]), base


class Predictor:
    """Loads the three PI-NGB-TSTR models and the app metadata."""
    def __init__(self, model_dir):
        self.models = {t: NGBEnsemble(os.path.join(model_dir, f'{t}.npz')) for t in TARGETS}
        self.meta = json.load(open(os.path.join(model_dir, 'meta.json')))

    def predict(self, raw, level=0.90):
        prep = prepare_inputs(raw)
        res = pd.DataFrame(index=raw.index)
        for t, m in self.models.items():
            mu = m.mean(m.design_matrix(prep)); lo, hi = m.interval(mu, level)
            res[f'{t}_pred'], res[f'{t}_lo'], res[f'{t}_hi'] = mu, lo, hi
            if t == 'TSR':
                sig = m.spec['calibration']['sigma']
                z = (TSR_LIMIT - mu) / sig
                p = 1 - 0.5 * (1 + np.vectorize(math.erf)(z / math.sqrt(2)))
                res['TSR_P_pass'] = p
                res['TSR_decision'] = np.where(p >= 0.90, 'Pass', np.where(p <= 0.10, 'Fail', 'Uncertain – test in laboratory'))
        return res, prep

    def checks(self, raw):
        return [check_inputs(r, self.meta['domain']) for r in raw.to_dict('records')]

    def explain(self, prep_row_df, target, n_perm=10):
        m = self.models[target]
        return m.shap(m.design_matrix(prep_row_df)[0], n_perm=n_perm)

In [ ]:
%%writefile rapcore/export.py
"""Export a trained PI-NGB-TSTR model (official ngboost NGBRegressor) to the app's portable format.

Run once where the model was trained (e.g. Colab, with scikit-learn and ngboost installed). The training
pipeline is reproduced exactly: physics descriptors, training-median / mode filling, one-hot encoding
of the categorical inputs with the first category dropped, numerical inputs passed through unscaled."""
import json
import numpy as np
import pandas as pd
from .features import prepare_inputs


def _training_frames(xlsx, target, drop_cols=()):
    tr, te, va = [pd.read_excel(xlsx, s) for s in ('Training', 'Testing', 'Validation')]
    tr, te, va = [d.drop(columns=['Original_sheet'], errors='ignore').dropna(subset=[target]) for d in (tr, te, va)]
    tr, te, va = prepare_inputs(tr), prepare_inputs(te), prepare_inputs(va)
    fill = {}
    for c in tr.columns:
        if c == target:
            continue
        if pd.api.types.is_numeric_dtype(tr[c]):
            fill[c] = float(tr[c].median())
        else:
            mv = tr[c].mode(); fill[c] = str(mv.iloc[0]) if len(mv) else None
    for d in (tr, te, va):
        for c, v in fill.items():
            if c in d and v is not None:
                d[c] = d[c].fillna(v)
        d.drop(columns=[c for c in drop_cols if c in d], inplace=True)
    return tr, te, va, fill


def _spec_from_training(Xtr):
    """Replicates ColumnTransformer([('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cats)], remainder='passthrough')."""
    cat_cols = [c for c in Xtr.columns if not pd.api.types.is_numeric_dtype(Xtr[c])]
    num_cols = [c for c in Xtr.columns if c not in cat_cols]
    cats = {c: sorted(Xtr[c].astype(str).unique().tolist()) for c in cat_cols}
    names = [f'cat__{c}_{v}' for c in cat_cols for v in cats[c][1:]] + [f'remainder__{c}' for c in num_cols]
    return dict(x_cols=list(Xtr.columns), cat_cols=cat_cols, num_cols=num_cols, categories=cats,
                drop_idx={c: 0 for c in cat_cols}, feature_names=names)


def _trees(model, n_features):
    L, R, F, TH, V, roots, tparam, tmult = [], [], [], [], [], [], [], []
    off, depth = 0, 0
    for stage, (models, s, cidx) in enumerate(zip(model.base_models, model.scalings, model.col_idxs)):
        cidx = np.asarray(cidx) if cidx is not None and len(np.atleast_1d(cidx)) else np.arange(n_features)
        for p, est in enumerate(models):
            t = est.tree_; n = t.node_count; depth = max(depth, int(t.max_depth))
            left, right = t.children_left.copy(), t.children_right.copy()
            leaf = left < 0; own = np.arange(n)
            left[leaf] = own[leaf]; right[leaf] = own[leaf]
            feat = np.where(leaf, 0, cidx[np.maximum(t.feature, 0)])
            thr = np.where(leaf, np.inf, t.threshold)
            L.append(left + off); R.append(right + off); F.append(feat); TH.append(thr); V.append(t.value[:, 0, 0])
            roots.append(off); tparam.append(p); tmult.append(model.learning_rate * float(s)); off += n
    return dict(left=np.concatenate(L).astype(np.int32), right=np.concatenate(R).astype(np.int32),
                feat=np.concatenate(F).astype(np.int32), thr=np.concatenate(TH).astype(np.float64),
                val=np.concatenate(V).astype(np.float64), roots=np.array(roots, np.int32),
                tree_param=np.array(tparam, np.int32), tree_mult=np.array(tmult, np.float64)), depth


def export(model, xlsx, target, out_path, drop_cols=(), source='', n_bg=50, seed=0):
    """Write models/<target>.npz. `model` is the fitted NGBRegressor; `xlsx` the TSTR data file it was trained on.
    Returns the held-out metrics (to compare with the paper) and the maximum difference between the exported
    model and model.predict on all data (should be ~1e-12)."""
    from .engine import NGBEnsemble
    tr, te, va, fill = _training_frames(xlsx, target, drop_cols)
    Xtr = tr.drop(columns=[target])
    spec = _spec_from_training(Xtr)
    dist = 'lognormal' if ('LogNormal' in str(getattr(model, 'Dist', '')) or
                           getattr(getattr(model, 'D', None), 'kind', '') == 'lognormal') else 'normal'
    trees, depth = _trees(model, len(spec['feature_names']))
    rng = np.random.default_rng(seed)
    spec.update(target=target, dist=dist, fill=fill, init_params=[float(v) for v in np.ravel(model.init_params)],
                max_depth=depth, source=source, dropped_inputs=list(drop_cols))
    tmp = dict(spec=json.dumps(dict(spec, calibration=dict(mode='absolute', scores=[0.0], sigma=1.0))), bg=np.zeros((1, 1)), **trees)
    np.savez_compressed(out_path, **tmp)
    eng = NGBEnsemble(out_path)
    mats = {k: eng.design_matrix(d) for k, d in (('Training', tr), ('Testing', te), ('Validation', va))}
    # exactness check against the original model on the same processed matrices
    diff = max(float(np.max(np.abs(eng.mean(M) - np.asarray(model.predict(M))))) for M in mats.values())
    ys = {k: d[target].values.astype(float) for k, d in (('Training', tr), ('Testing', te), ('Validation', va))}
    preds = {k: eng.mean(M) for k, M in mats.items()}
    y_ho = np.r_[ys['Testing'], ys['Validation']]; p_ho = np.r_[preds['Testing'], preds['Validation']]
    mode = 'relative' if target == 'RD' else 'absolute'
    scores = np.abs(y_ho - p_ho) / (np.abs(p_ho) if mode == 'relative' else 1.0)
    cal = dict(mode=mode, scores=sorted(float(v) for v in scores), sigma=float(np.sqrt(np.mean((y_ho - p_ho) ** 2))))
    metrics = {}
    for k in ('Training', 'Testing', 'Validation'):
        y, p = ys[k][:132] if k == 'Training' else ys[k], preds[k][:132] if k == 'Training' else preds[k]
        metrics[k] = dict(R2=float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)), RMSE=float(np.sqrt(np.mean((y - p) ** 2))),
                          MAPE=float(100 * np.mean(np.abs(y - p) / np.abs(y))), n=int(len(y)))
    bg = mats['Training'][rng.choice(132, size=min(n_bg, 132), replace=False)]
    spec.update(calibration=cal, metrics=metrics)
    np.savez_compressed(out_path, spec=json.dumps(spec), bg=bg, **trees)
    return metrics, diff

## 3. Upload the three TSTR data files

In [ ]:
import glob
need = {'ITSD': 'TSTR_ITSD', 'TSR': 'TSTR_TSR', 'RD': 'TSTR_RD'}
def find():
    return {t: next((f for f in glob.glob('*.xlsx') if k in f.replace(' ', '_')), None) for t, k in need.items()}
FILES = find()
if not all(FILES.values()):
    from google.colab import files
    files.upload()
    FILES = find()
print(FILES)
assert all(FILES.values()), 'Upload all three data files.' 

## 4. Train the three models (settings copied from your PI-NGB-TSTR notebooks)

In [ ]:
import numpy as np, pandas as pd, pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
from ngboost import NGBRegressor
from ngboost.distns import Normal, LogNormal
from ngboost.scores import MLE
from rapcore.export import _training_frames, export
from rapcore.engine import NGBEnsemble

SETTINGS = {
    'ITSD': dict(base=DecisionTreeRegressor(max_depth=6, min_samples_leaf=13, random_state=42), Dist=Normal,
                 n_estimators=1500, learning_rate=0.025, minibatch_frac=1.0, col_sample=1.0, random_state=42, drop=[], weight=None),
    'TSR':  dict(base=ExtraTreeRegressor(max_depth=15, min_samples_leaf=4, max_features=0.7, random_state=7), Dist=Normal,
                 n_estimators=80, learning_rate=0.05, minibatch_frac=0.9, col_sample=1.0, random_state=63,
                 drop=['AT', 'RAP', 'RCT', 'Ndes'], weight=(132, 3.0)),
    'RD':   dict(base=DecisionTreeRegressor(max_depth=6, min_samples_leaf=8, random_state=42), Dist=LogNormal,
                 n_estimators=400, learning_rate=0.03, minibatch_frac=0.3, col_sample=0.85, random_state=42, drop=[], weight=None)}
PAPER = {'ITSD': (0.8327, 0.8166), 'TSR': (0.8270, 0.8012), 'RD': (0.8575, 0.8289)}   # testing / validation R2 in the paper

MODELS = {}
for t, s in SETTINGS.items():
    tr, te, va, _ = _training_frames(FILES[t], t, s['drop'])
    Xtr, Xte, Xva = (d.drop(columns=[t]) for d in (tr, te, va))
    cats = Xtr.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    pre = ColumnTransformer([('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cats)], remainder='passthrough')
    A, B, C = pre.fit_transform(Xtr), pre.transform(Xte), pre.transform(Xva)
    w = None
    if s['weight']:
        w = np.ones(len(A)); w[:s['weight'][0]] = s['weight'][1]
    model = NGBRegressor(Base=s['base'], Dist=s['Dist'], Score=MLE, natural_gradient=True, n_estimators=s['n_estimators'],
                         learning_rate=s['learning_rate'], minibatch_frac=s['minibatch_frac'], col_sample=s['col_sample'],
                         random_state=s['random_state'], verbose=False)
    model.fit(A, tr[t].values, sample_weight=w) if w is not None else model.fit(A, tr[t].values)
    r2 = [r2_score(d[t], model.predict(M)) for d, M in ((te, B), (va, C))]
    print(f'{t}: testing R2 = {r2[0]:.4f} (paper {PAPER[t][0]:.4f}) | validation R2 = {r2[1]:.4f} (paper {PAPER[t][1]:.4f})')
    MODELS[t] = (model, pre, (A, B, C), s['drop'])

## 5. Export for the app and verify

In [ ]:
os.makedirs('models_official', exist_ok=True)
src = f'official ngboost {ngboost.__version__}, scikit-learn {sklearn.__version__} (PI-NGB-TSTR notebooks)'
for t, (model, pre, mats, drop) in MODELS.items():
    out = f'models_official/{t}.npz'
    metrics, diff = export(model, FILES[t], t, out, drop_cols=drop, source=src)
    eng = NGBEnsemble(out)
    tr, te, va, _ = _training_frames(FILES[t], t, drop)
    same = max(np.abs(eng.design_matrix(d) - M).max() for d, M in zip((tr, te, va), mats))
    print(f"{t}: preprocessing identical to scikit-learn: {same == 0} | exported vs model.predict max diff {diff:.1e} | "
          f"test R2 {metrics['Testing']['R2']:.4f}, val R2 {metrics['Validation']['R2']:.4f}")
    names = {'ITSD': 'ITSD', 'TSR': 'TSR', 'RD': 'TSTR_RD'}[t]
    pickle.dump(model, open(f'NGB_PhysicsInformed_{names}_Model.pkl', 'wb'))
    pickle.dump(pre, open(f'NGB_PhysicsInformed_{names}_Preprocessor.pkl', 'wb'))

## 6. Download

In [ ]:
import shutil, zipfile
shutil.make_archive('models_official', 'zip', 'models_official')
with zipfile.ZipFile('PI-NGB_TSTR_pickles.zip', 'w') as z:
    for f in sorted(glob.glob('NGB_PhysicsInformed_*.pkl')): z.write(f)
try:
    from google.colab import files
    files.download('models_official.zip'); files.download('PI-NGB_TSTR_pickles.zip')
except ImportError:
    print('Files are in the working folder.')